# Sigiriya Hybrid GraphRAG — Local Retrieval Evaluation (Before SinLlama)

**Goal:** Test local Neo4j vector + graph retrieval **without SinLlama** using questions grounded in the supplied *සීගිරිය – සංචාරක තොරතුරු* document. Includes 22 section-coverage questions, multi-section queries, and unsupported/live-data queries.

**This notebook is read-only:** it does not change the database, generate embeddings, or call a language model. It uses the group's original retrieval modules and compares ranked chunk IDs with independently recorded expected source sections. Results remain **unverified** until manually checked for factual sufficiency. The evaluation labels are only for measuring retrieval, never passed into retrieval functions.

**Location:** save this notebook in `Sigiriya-Hybrid-GraphRAG/notebooks/`. Group scripts should be in `original_group_work/` (either directly or in its extracted `Sigiriya_Hybrid_GraphRAG_Stage2/` subfolder). Select your Sigiriya `.venv` as the notebook kernel in VS Code.

## 1 — Requirements

Start the **Sigiriya Neo4j Desktop instance** and confirm it is the one listening on `bolt://127.0.0.1:7687`. Python environment dependencies: `neo4j`, `sentence-transformers`, `torch`, `pandas`, `ipykernel` (install via your root `requirements.txt`). Run in VS Code; do not paste your password into code or source control.

In [ ]:
from pathlib import Path
import os, sys, json, csv, time, getpass, importlib
import pandas as pd
from IPython.display import display
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer

# Locate group's original Python files relative to VS Code's working directory.
starts = [Path.cwd(), *list(Path.cwd().parents)[:4]]
locations = []
for root in starts:
    locations.extend([
        root, root / "original_group_work",
        root / "original_group_work" / "Sigiriya_Hybrid_GraphRAG_Stage2",
        root / "Sigiriya_Hybrid_GraphRAG_Stage2"
    ])
GROUP_DIR = next((p.resolve() for p in locations if (p / "Sigiriya_GraphRAG_Stage2.py").is_file()), None)
if GROUP_DIR is None:
    raise FileNotFoundError("Cannot find Sigiriya_GraphRAG_Stage2.py. Check original_group_work folder and notebook working directory.")
sys.path.insert(0, str(GROUP_DIR))
PROJECT_ROOT = next((p for p in [GROUP_DIR.parent, GROUP_DIR.parent.parent] if (p / 'results').is_dir()), GROUP_DIR.parent)
OUTPUT_DIR = PROJECT_ROOT / "results" / "retrieval"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Python:', sys.executable)
print('Group scripts:', GROUP_DIR)
print('Results folder:', OUTPUT_DIR)

## 2 — Connect to local Neo4j and load E5 only once

Set `NEO4J_URI` if your new Sigiriya instance uses a different local Bolt port. The password is read from a hidden input; alternatively supply it as an environment variable. **Do not point this notebook at your earlier Ella instance.**

In [ ]:
import Sigiriya_Multi_Query_Retrieval as multi
import Sigiriya_GraphRAG_Stage2 as stage2
import Sigiriya_GraphRAG_Stage2_Answer_Refinement as refinement
from Sigiriya_Retrieval_Improvement_Batch_Reviewed import load_sources

URI = os.getenv('NEO4J_URI', 'bolt://127.0.0.1:7687')
USERNAME = os.getenv('NEO4J_USERNAME', 'neo4j')
DATABASE = os.getenv('NEO4J_DATABASE', 'neo4j')
PASSWORD = os.getenv('NEO4J_PASSWORD') or getpass.getpass('Sigiriya Neo4j password: ')

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))
driver.verify_connectivity()
with driver.session(database=DATABASE) as session:
    rows = session.run("MATCH (c:Chunk {destination:'Sigiriya'}) RETURN count(c) AS n, count(c.embedding) AS vectors").single()
    entities = session.run("MATCH (e:TourismEntity {scope:'Sigiriya'}) RETURN count(e) AS n").single()['n']
    indexes = {r['name']:r['state'] for r in session.run('SHOW VECTOR INDEXES YIELD name,state RETURN name,state')}
assert rows['n'] == 22 and rows['vectors'] == 22 and entities == 49, f'Wrong or incomplete database: chunks={rows["n"]}, vectors={rows["vectors"]}, entities={entities}'
for index in (multi.TEXT_INDEX, multi.HEADING_INDEX):
    assert indexes.get(index) == 'ONLINE', f'{index} not ONLINE: {indexes.get(index)}'
print('Connected:', URI, '| chunks:', rows['n'], '| entities:', entities)
print('Indexes:', indexes)

# Neo4j Desktop compatibility: group's original SEARCH form may require a newer release.
# Here we use db.index.vector.queryNodes without changing the team's source files.
def local_vector_search(session, index_name, vector):
    if index_name not in (multi.TEXT_INDEX, multi.HEADING_INDEX):
        raise ValueError('Unexpected vector index')
    cypher = ("CALL db.index.vector.queryNodes($index, $k, $vector) "
              "YIELD node, score WHERE node:Chunk AND node.destination = 'Sigiriya' "
              "RETURN node.id AS chunk_id, score ORDER BY score DESC")
    return [r['chunk_id'] for r in session.run(cypher, index=index_name, k=22, vector=vector)]
multi.search_index = local_vector_search
print('Loading E5 model:', multi.MODEL_NAME)
model = SentenceTransformer(multi.MODEL_NAME, device='cpu')
manifest = stage2.load_manifest(GROUP_DIR / 'Sigiriya_KG_Expansion_Manifest.json')
with driver.session(database=DATABASE) as session:
    headings = {r['chunk_id']:r['heading'] for r in session.run(multi.HEADINGS_QUERY)}
    sources = load_sources(session)
print('Ready to test; source chunks:', len(sources))

## 3 — Question set (32 tests)

The first **22** questions cover all sections of your uploaded Sinhala knowledge base. The next **7** require multiple sections. The final **3** are not answerable with reliable static document evidence. For section-based questions, `expected_ids` identifies relevant source chunks (e.g. `sigiriya_006`), **not a predefined model answer**.

**Important:** Current prices and opening times in the document are historical/source-supplied information; the system must **not** present them as today's verified values. Labels are based on the provided document, not independently verified facts about Sigiriya.

In [ ]:
TESTS = [{'id': 'Q01', 'type': 'single', 'question': 'සීගිරිය පිහිටා ඇත්තේ කුමන පළාතේ සහ දිස්ත්\u200dරික්කයේද?', 'expected_ids': ['sigiriya_001'], 'answerability': 'supported'}, {'id': 'Q02', 'type': 'single', 'question': 'සීගිරිය රාජකීය අගනුවරක් ලෙස සංවර්ධනය කළේ කවර රජුද?', 'expected_ids': ['sigiriya_002'], 'answerability': 'supported'}, {'id': 'Q03', 'type': 'single', 'question': 'සීගිරි පර්වතයේ මුදුනේ සිට දැකගත හැක්කේ මොනවාද?', 'expected_ids': ['sigiriya_003'], 'answerability': 'supported'}, {'id': 'Q04', 'type': 'single', 'question': 'සීගිරි බිතුසිතුවම්වල දැක්වෙන්නේ කවර රූපද?', 'expected_ids': ['sigiriya_004'], 'answerability': 'supported'}, {'id': 'Q05', 'type': 'single', 'question': 'කැටපත් පවුර මත පැරණි සංචාරකයන් ලියා තිබුණේ මොනවාද?', 'expected_ids': ['sigiriya_005'], 'answerability': 'supported'}, {'id': 'Q06', 'type': 'single', 'question': 'සීගිරියේ ජල උද්\u200dයානවල ජලය පාලනයට භාවිත කළේ මොනවාද?', 'expected_ids': ['sigiriya_006'], 'answerability': 'supported'}, {'id': 'Q07', 'type': 'single', 'question': 'සීගිරියේ දිය උල්පත් ක්\u200dරියාත්මක වීමට භාවිත වූ බලවේග මොනවාද?', 'expected_ids': ['sigiriya_007'], 'answerability': 'supported'}, {'id': 'Q08', 'type': 'single', 'question': 'සීගිරියේ ගල් උද්\u200dයානයේ දැකගත හැකි අංග මොනවාද?', 'expected_ids': ['sigiriya_008'], 'answerability': 'supported'}, {'id': 'Q09', 'type': 'single', 'question': 'නාග පෙන ගුහාවට එම නම ලැබුණේ ඇයි?', 'expected_ids': ['sigiriya_009'], 'answerability': 'supported'}, {'id': 'Q10', 'type': 'single', 'question': 'සීගිරි කෞතුකාගාරයෙන් දැනගත හැකි කරුණු මොනවාද?', 'expected_ids': ['sigiriya_010'], 'answerability': 'supported'}, {'id': 'Q11', 'type': 'single', 'question': 'ලේඛනයේ සඳහන් දකුණු ආසියානු සංචාරක ප්\u200dරවේශ ගාස්තුව කීයද?', 'expected_ids': ['sigiriya_011'], 'answerability': 'supported'}, {'id': 'Q12', 'type': 'single', 'question': 'සීගිරියේදී සංචාරකයන්ට කළ හැකි ක්\u200dරියාකාරකම් තුනක් මොනවාද?', 'expected_ids': ['sigiriya_012'], 'answerability': 'supported'}, {'id': 'Q13', 'type': 'single', 'question': 'පිදුරංගල නැගීමේදී අවසාන කොටසේ අවධානය යොමු කළ යුත්තේ මොනවාටද?', 'expected_ids': ['sigiriya_013'], 'answerability': 'supported'}, {'id': 'Q14', 'type': 'single', 'question': 'Mapagala පිහිටා ඇත්තේ සීගිරියට කුමන දිශාවෙන්ද?', 'expected_ids': ['sigiriya_014'], 'answerability': 'supported'}, {'id': 'Q15', 'type': 'single', 'question': 'සීගිරිය ආසන්නයේ වනජීවී සෆාරි සඳහා සඳහන් ජාතික වනෝද්\u200dයාන මොනවාද?', 'expected_ids': ['sigiriya_015'], 'answerability': 'supported'}, {'id': 'Q16', 'type': 'single', 'question': 'මහනුවර සිට සීගිරියට යාමට ලේඛනයේ සඳහන් මාර්ගය කුමක්ද?', 'expected_ids': ['sigiriya_016'], 'answerability': 'supported'}, {'id': 'Q17', 'type': 'single', 'question': 'සීගිරිය පර්වතය තරණය කිරීමට පෙර රැගෙන යා යුතු වැදගත් දෙය කුමක්ද?', 'expected_ids': ['sigiriya_017'], 'answerability': 'supported'}, {'id': 'Q18', 'type': 'single', 'question': 'සීගිරිය හා හබරණ අවට සඳහන් නවාතැන් වර්ග මොනවාද?', 'expected_ids': ['sigiriya_018'], 'answerability': 'supported'}, {'id': 'Q19', 'type': 'single', 'question': 'සීගිරිය නැරඹීමට උදෑසන පැමිණීම සුදුසු වන්නේ ඇයි?', 'expected_ids': ['sigiriya_019'], 'answerability': 'supported'}, {'id': 'Q20', 'type': 'single', 'question': 'වැසි සහිත දිනක සීගිරිය තරණය කිරීමේදී ඇති අවදානම් මොනවාද?', 'expected_ids': ['sigiriya_020'], 'answerability': 'supported'}, {'id': 'Q21', 'type': 'single', 'question': 'සීගිරි පර්වතයේ බඹර කූඩු අසල හැසිරිය යුත්තේ කෙසේද?', 'expected_ids': ['sigiriya_021'], 'answerability': 'supported'}, {'id': 'Q22', 'type': 'single', 'question': 'සීගිරියේ පුරාවිද්\u200dයාත්මක උරුමය රැක ගැනීමට සංචාරකයන් කළ යුතු දේ මොනවාද?', 'expected_ids': ['sigiriya_022'], 'answerability': 'supported'}, {'id': 'Q23', 'type': 'multi', 'question': 'සීගිරියේ ජල උද්\u200dයාන සහ දිය උල්පත් උද්\u200dයානයේ ජල පද්ධති විස්තර කරන්න.', 'expected_ids': ['sigiriya_006', 'sigiriya_007'], 'answerability': 'supported'}, {'id': 'Q24', 'type': 'multi', 'question': 'සීගිරි බිතුසිතුවම් සහ කැටපත් පවුරේ ඓතිහාසික වැදගත්කම පැහැදිලි කරන්න.', 'expected_ids': ['sigiriya_004', 'sigiriya_005'], 'answerability': 'supported'}, {'id': 'Q25', 'type': 'multi', 'question': 'සීගිරියේ ජල උද්\u200dයානය සහ ගල් උද්\u200dයානය අතර වෙනස්කම් මොනවාද?', 'expected_ids': ['sigiriya_006', 'sigiriya_008'], 'answerability': 'supported'}, {'id': 'Q26', 'type': 'multi', 'question': 'සීගිරිය සහ පිදුරංගල තරණය කිරීමේදී අවශ්\u200dය සූදානම ගැන කියන්න.', 'expected_ids': ['sigiriya_003', 'sigiriya_013', 'sigiriya_020'], 'answerability': 'supported'}, {'id': 'Q27', 'type': 'multi', 'question': 'සීගිරියට යන මාර්ග සහ නැරඹීමට සුදුසු වේලාව පැහැදිලි කරන්න.', 'expected_ids': ['sigiriya_016', 'sigiriya_019'], 'answerability': 'supported'}, {'id': 'Q28', 'type': 'multi', 'question': 'සීගිරියේ බඹර අනතුරු සහ ඓතිහාසික ස්ථාන රැකගැනීම ගැන උපදෙස් දෙන්න.', 'expected_ids': ['sigiriya_021', 'sigiriya_022'], 'answerability': 'supported'}, {'id': 'Q29', 'type': 'multi', 'question': 'සීගිරිය අවට නවාතැන් සහ වනජීවී සෆාරි අවස්ථා ගැන විස්තර කරන්න.', 'expected_ids': ['sigiriya_018', 'sigiriya_015'], 'answerability': 'supported'}, {'id': 'Q30', 'type': 'unsupported', 'question': 'සීගිරියේ අද ප්\u200dරවේශ පත්\u200dර මිල කීයද?', 'expected_ids': [], 'answerability': 'live_unsupported'}, {'id': 'Q31', 'type': 'unsupported', 'question': 'සීගිරියේ මේ මොහොතේ කාලගුණය කෙසේද?', 'expected_ids': [], 'answerability': 'live_unsupported'}, {'id': 'Q32', 'type': 'unsupported', 'question': 'සීගිරියේ 2035 වසරේ ප්\u200dරවේශ ගාස්තුව කීයද?', 'expected_ids': [], 'answerability': 'unsupported'}]

question_df = pd.DataFrame(TESTS)
display(question_df[['id','type','question','expected_ids','answerability']])
print('Questions:',len(TESTS), '| single:',sum(t['type']=='single' for t in TESTS),'| multi:',sum(t['type']=='multi' for t in TESTS),'| unsupported:',sum(t['type']=='unsupported' for t in TESTS))

## 4 — One complete retrieval test (no SinLlama)

Calls the same E5 vector fusion, multi-query and graph-traversal ranking as the group's pipeline. Also obtains the group's conservative source-extractive answer. Expected IDs are used **only after retrieval**, for comparison. `top3` checks whether evidence needed to answer the question was found, not whether generated wording is correct.

In [ ]:
def retrieve_one(test):
    q = test['question']
    t0 = time.perf_counter()
    with driver.session(database=DATABASE) as session:
        # Grounded extractive answer (same as your original export script)
        extracted = refinement.quality_answer(q, session, model, sources, headings)
        status = extracted.get('status', 'UNKNOWN')
        if status == 'LIVE_DATA_UNSUPPORTED':
            original, multi_rank, hybrid, paths, anchors = [], [], [], [], []
        else:
            original = multi.fused_search(session, model, q)
            _, multi_rank = multi.multi_query_search(session, model, q, headings)
            anchors = refinement.resolve_anchors(session, q, manifest['entities'], original)
            paths = refinement.graph_paths(session, q, manifest['entities'], anchors, limit=12)
            hybrid, _ = refinement.rank_chunks(original, multi_rank, paths)
        expected = set(test['expected_ids'])
        top3 = hybrid[:3]
        hits = expected.intersection(top3)
        return {
            'id': test['id'], 'type': test['type'], 'question': q,
            'answerability': test['answerability'], 'expected_ids': test['expected_ids'],
            'status': status, 'extractive_answer': extracted.get('answer', ''),
            'vector_top3': original[:3], 'multi_query_top3': multi_rank[:3],
            'hybrid_top3': top3, 'graph_path_count': len(paths),
            'graph_paths': paths[:4], 'anchors': anchors,
            'hits_at_3': len(hits),
            'all_expected_at_3': (len(hits)==len(expected)) if expected else None,
            'evidence_recall_at_3': len(hits)/len(expected) if expected else None,
            'latency_s': round(time.perf_counter()-t0, 3),
            'retrieved_evidence': [{'id':cid, 'heading':sources[cid]['heading'],
                                    'text':sources[cid]['evidence']}
                                   for cid in top3 if cid in sources],
        }

# Start with the known water-garden test, then proceed to all questions.
SMOKE_TEST = next(t for t in TESTS if t['id']=='Q23')
smoke = retrieve_one(SMOKE_TEST)
for k in ('id','question','status','expected_ids','vector_top3','multi_query_top3','hybrid_top3','graph_path_count','all_expected_at_3','extractive_answer','latency_s'):
    print(f'{k}: {smoke[k]}')
print('\nTOP SOURCE EVIDENCE:')
for s in smoke['retrieved_evidence']:
    print(f"\n[{s['id']}] {s['heading']}\n{s['text'][:600]}")

## 5 — Run all 32 questions (or start with a subset)

For a quick first batch, change `RUN_COUNT = 5`. For the complete evaluation, leave it as `len(TESTS)`. Execution may take time on a CPU because each question performs several embedding searches. Error rows are retained so one failure does not discard earlier tests.

In [ ]:
RUN_COUNT = len(TESTS)  # e.g., 5 for a quick trial; then run all 32
results = []
for i, test in enumerate(TESTS[:RUN_COUNT], start=1):
    try:
        r = retrieve_one(test)
        results.append(r)
        print(f"{i:02}/{RUN_COUNT} {test['id']} {r['status']} top3={r['hybrid_top3']} recall={r['evidence_recall_at_3']}")
    except Exception as exc:
        results.append({**test, 'status':'ERROR', 'error':str(exc),
                        'hybrid_top3':[], 'graph_path_count':None,
                        'all_expected_at_3':None, 'evidence_recall_at_3':None})
        print(f"{i:02}/{RUN_COUNT} {test['id']} ERROR: {type(exc).__name__}: {exc}")

## 6 — Summary and review table

**Evidence Recall@3** = share of manually identified relevant source sections found in the hybrid top three (for supported questions). **All Evidence@3** means every expected section is present. For unsupported questions, manually verify that the answer refuses to invent unsupported facts. These metrics **do not** establish correctness of the extractive wording or completeness of the knowledge graph. The status values are generated by the group's heuristic, not independent truth labels.

In [ ]:
summary = pd.DataFrame([{k:r.get(k) for k in (
    'id','type','question','answerability','status','expected_ids','vector_top3',
    'multi_query_top3','hybrid_top3','hits_at_3','evidence_recall_at_3',
    'all_expected_at_3','graph_path_count','latency_s','extractive_answer','error')}
    for r in results])
for col in ('expected_ids','vector_top3','multi_query_top3','hybrid_top3'):
    if col in summary:
        summary[col] = summary[col].apply(lambda x: ', '.join(x) if isinstance(x, list) else x)
if len(summary):
    supported = pd.DataFrame([r for r in results if r.get('expected_ids')])
    print('Tests processed:', len(results), '| errors:', sum(r.get('status')=='ERROR' for r in results))
    if not supported.empty:
        print('Mean evidence Recall@3:', round(supported['evidence_recall_at_3'].dropna().mean(),3))
        print('All expected chunks within top 3:', int(supported['all_expected_at_3'].sum()),'/',len(supported))
    display(summary[['id','type','status','expected_ids','hybrid_top3','evidence_recall_at_3','graph_path_count','extractive_answer']])

## 7 — Manual review fields and export

The output CSV includes blank columns for human assessment: **evidence_relevant**, **answer_grounded**, **complete**, and **notes**. Mark these manually after checking the complete source excerpts. A supported question with a good top-3 rank may still have an incomplete extractive answer. Unsupported/live questions should not be scored using Recall@3.

Files are saved under `results/retrieval/` and can be committed to GitHub **after you remove sensitive data if any**. No secrets are saved by this notebook.

In [ ]:
for col in ('evidence_relevant','answer_grounded','complete','review_notes'):
    summary[col] = ''
CSV_PATH = OUTPUT_DIR / 'sigiriya_32_question_retrieval_review.csv'
JSON_PATH = OUTPUT_DIR / 'sigiriya_32_question_retrieval_details.json'
summary.to_csv(CSV_PATH, index=False, encoding='utf-8-sig')
JSON_PATH.write_text(json.dumps(results, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
print('Saved CSV:', CSV_PATH)
print('Saved JSON:', JSON_PATH)

## 8 — Optional: ask your own question (read-only)

This cell retrieves results for a custom question without altering the 32-question evaluation or its metrics. The expected IDs are intentionally blank, so gold metrics are not computed. This is **not** a SinLlama response.

In [ ]:
CUSTOM_QUESTION = 'සීගිරියේ කැටපත් පවුරේ ඇති සටහන්වල වැදගත්කම කුමක්ද?'
custom = retrieve_one({'id':'CUSTOM','type':'custom','question':CUSTOM_QUESTION,
                       'expected_ids':[], 'answerability':'unknown'})
print('Status:', custom['status'])
print('Extractive answer:', custom['extractive_answer'])
print('Top3:', custom['hybrid_top3'])
print('Graph paths:', custom['graph_path_count'])
for ev in custom['retrieved_evidence']:
    print(f"\n{ev['id']} — {ev['heading']}\n{ev['text'][:700]}")

## 9 — Close Neo4j connection

Run when finished (or restart the notebook kernel).

In [ ]:
driver.close()
print('Neo4j connection closed.')